# 07 — Autonomous goal-seeking agent

**Definition:** I give a **goal**, not a task. The agent owns the loop until the goal is
measurably met — or the budget runs out.

Ancestors: AutoGPT, BabyAGI. Modern form: long-running research / "deep" agents.

```
      GOAL (+ success criteria)
        |
        v
   +----------+
+->| OBSERVE  |  what's the current world state?
|  +----------+
|        v
|  +----------+
|  |  PLAN    |  what should I do next, given the goal?
|  +----------+
|        v
|  +----------+
|  |   ACT    |  execute
|  +----------+
|        v
|  +----------+
|  | EVALUATE |  AM I DONE? measured against the criteria
|  +----------+
|        |
+--------+ not yet
         |
         +--- goal met / budget spent ---> DONE
```

**What actually makes this different from 1–6:**

| | patterns 1–6 | pattern 7 |
|---|---|---|
| human gives | a **task** | a **goal** |
| who owns the loop | the human, turn by turn | **the agent** |
| stops when | the task is done | the **goal is measurably met** |
| success is | implicit | **defined explicitly, upfront** |
| runs for | seconds | minutes to hours |
| needs | — | budgets, guardrails, HITL |

The defining component is the **evaluator** — a node that answers "are we there yet?"
against criteria fixed *before* the run. Without one this is just ReAct with a big
recursion limit.

And the honest version of that: an agent that cannot say *"I could not achieve this goal"*
isn't autonomous, it's a hallucination machine with extra steps. Hence three exits, not two.

## Setup

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

# Both providers serve the SAME model (gpt-oss-120b), so behaviour is identical.
# Groq is the default; set LLM_PROVIDER=cerebras to switch. I added that second path
# after burning through Groq's 200k-tokens-per-day cap while writing these notebooks.
if os.environ.get("LLM_PROVIDER", "groq") == "cerebras":
    from langchain_openai import ChatOpenAI

    if not os.environ.get("CEREBRAS_API_KEY"):
        os.environ["CEREBRAS_API_KEY"] = getpass.getpass("CEREBRAS_API_KEY: ")
    llm = ChatOpenAI(
        model="gpt-oss-120b",
        temperature=0,
        base_url="https://api.cerebras.ai/v1",
        api_key=os.environ["CEREBRAS_API_KEY"],
        timeout=120,      # ChatOpenAI defaults to NO timeout - a stalled
        max_retries=5,    # connection hangs the whole notebook forever
    )
else:
    from langchain_groq import ChatGroq

    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")
    # reasoning_format="hidden" keeps gpt-oss's chain-of-thought out of .content
    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0,
        reasoning_format="hidden",
        timeout=120,
        max_retries=5,
    )


def show(graph):
    print(graph.get_graph().draw_mermaid())


print(llm.invoke("Reply with the single word: ready").content)

# Part A — the goal-seeking loop

## Tools — read-only on purpose

Safety principle: an autonomous agent gets **read-only** tools by default. Anything with a
side effect goes behind a human gate (Part C).

In [ ]:
from langchain_core.tools import tool


@tool
def search(query: str) -> str:
    """Search a knowledge base for information about a topic."""
    kb = {
        "kafka": "Kafka: distributed log, ~1M msg/s per broker, p99 ~5ms, "
                 "durable via replication factor 3.",
        "rabbitmq": "RabbitMQ: AMQP broker, ~50k msg/s, richer routing, "
                    "weaker at replay/retention.",
        "pulsar": "Pulsar: segmented storage via BookKeeper, tiered offload, "
                  "~1M msg/s, native multi-tenancy.",
        "latency": "Latency: Kafka p99 5ms; RabbitMQ p99 1ms at low volume but "
                   "degrades sharply past 50k msg/s.",
        "durability": "Durability: Kafka replication + ISR; Pulsar quorum writes; "
                      "RabbitMQ mirrored queues (costly).",
    }
    hits = [v for k, v in kb.items() if k in query.lower()]
    return "\n".join(hits) if hits else f"No results for '{query}'."


@tool
def list_topics() -> str:
    """List every topic available in the knowledge base."""
    return "kafka, rabbitmq, pulsar, latency, durability"


tools = [search, list_topics]

## State

The field taxonomy matters more than it looks:

```python
goal, success_criteria   # WHAT and WHEN-to-stop   -> IMMUTABLE, the anchor
plan                     # HOW                     -> replaced every cycle
findings                 # WHAT WE LEARNED         -> accumulates (needs a reducer)
iterations, max_iterations  # BUDGET               -> the safety rail
goal_met, confidence     # THE VERDICT
final_report             # THE OUTPUT
```

**Rule:** `goal` and `success_criteria` are immutable. If the agent can rewrite its own
definition of success, it will always "succeed". Everything that legitimately drifts (plan,
findings) is kept separate from the anchor.

In [ ]:
import operator
from typing import Annotated, List, TypedDict

from pydantic import BaseModel, Field


class AutoState(TypedDict):
    # === IMMUTABLE - the anchor ===
    goal: str
    success_criteria: List[str]

    # === MUTABLE - the work ===
    plan: List[str]                              # replaced each cycle
    findings: Annotated[List[str], operator.add]  # ACCUMULATES
    observations: str                            # current world snapshot

    # === BUDGET - the safety rail ===
    iterations: int
    max_iterations: int

    # === OUTCOME ===
    goal_met: bool
    confidence: int
    final_report: str

## OBSERVE

A separate node so the plan is grounded in *what I actually have*, not what the LLM imagines
I have. Deterministic, no LLM call, costs nothing.

In [ ]:
def observe(state: AutoState) -> dict:
    findings = state.get("findings", [])
    obs = (
        f"Iteration: {state.get('iterations', 0)}/{state['max_iterations']}\n"
        f"Findings collected: {len(findings)}\n"
        + ("\n".join(f"  - {f[:150]}" for f in findings) if findings else "  (none yet)")
    )
    print(f"\nOBSERVE - {len(findings)} finding(s)")
    return {"observations": obs}

## PLAN

Regenerated **every cycle** from the immutable goal plus current observations. That
re-anchoring is what stops a long run from drifting off.

In [ ]:
class NextActions(BaseModel):
    """A small batch of concrete next steps."""

    actions: List[str] = Field(
        description="1-3 concrete next actions, each executable with the available tools. "
        "Focus on the biggest remaining gap."
    )
    rationale: str = Field(description="One sentence: why these actions now?")


planner = llm.with_structured_output(NextActions, method="json_schema")


def plan(state: AutoState) -> dict:
    prompt = (
        f"GOAL: {state['goal']}\n\n"
        "SUCCESS CRITERIA:\n" + "\n".join(f"- {c}" for c in state["success_criteria"])
        + f"\n\nCURRENT STATE:\n{state['observations']}\n\n"
        "Available tools: search(query), list_topics()\n\n"
        "What are the next 1-3 actions to close the biggest gap toward the goal? "
        "Do not repeat work already reflected in the findings."
    )
    result = planner.invoke(prompt)
    print(f"PLAN: {result.rationale}")
    for a in result.actions:
        print(f"   - {a}")
    return {"plan": result.actions}     # no reducer -> REPLACES the old plan

## ACT

A ReAct sub-agent again. Composition all the way down:
**Autonomous = Planning + ReAct + Reflection + Memory + Budget.**

In [ ]:
from langchain.agents import create_agent

executor = create_agent(
    model=llm,
    tools=tools,
    system_prompt="Execute the given actions using the tools. Report ONLY factual findings "
    "from tool output. If a tool returns nothing useful, say so plainly - never invent facts.",
)


def act(state: AutoState) -> dict:
    actions = "\n".join(f"{i}. {a}" for i, a in enumerate(state["plan"], 1))
    print("ACT")
    res = executor.invoke({"messages": [("user", f"Execute these actions:\n{actions}")]})
    finding = res["messages"][-1].content
    print(f"   -> {finding[:130]}...")

    # operator.add appends. Bumping iterations here is the budget clock ticking.
    return {"findings": [finding], "iterations": state.get("iterations", 0) + 1}

## EVALUATE — this node *is* the pattern

Everything else is patterns 2 and 3 recombined. The evaluator judges against the **fixed
criteria**, not against "does this look nice".

**Why not just ask the LLM if it's done?** Because LLMs are optimists — ask "are you done?"
and you get "yes!" far too early. Fixes, strongest first:

1. **deterministic criteria** — `len(findings) >= 5 and all(f.has_source)`
2. **structured verdict** with a numeric confidence and a threshold (what I do below)
3. an evaluator **with tools**, so it can verify claims
4. always: a hard budget as a backstop

In [ ]:
class GoalCheck(BaseModel):
    """A structured verdict - makes 'done' a threshold, not an opinion."""

    goal_met: bool = Field(description="True ONLY if EVERY success criterion is satisfied")
    confidence: int = Field(description="Confidence 1-10 that the goal is genuinely met")
    unmet_criteria: List[str] = Field(
        description="Criteria still NOT satisfied. Empty list if all are met."
    )
    reasoning: str = Field(description="One sentence justification.")


evaluator = llm.with_structured_output(GoalCheck, method="json_schema")


def evaluate(state: AutoState) -> dict:
    findings = "\n\n".join(state["findings"])
    prompt = (
        f"GOAL: {state['goal']}\n\n"
        "SUCCESS CRITERIA:\n" + "\n".join(f"- {c}" for c in state["success_criteria"])
        + f"\n\nEVIDENCE GATHERED:\n{findings}\n\n"
        "Be STRICT. Mark goal_met=True only if every single criterion is clearly satisfied "
        "by the evidence above. Partial coverage is NOT met."
    )
    check = evaluator.invoke(prompt)

    print(f"EVALUATE: met={check.goal_met} conf={check.confidence}/10 - {check.reasoning}")
    for u in check.unmet_criteria:
        print(f"   still missing: {u}")

    return {"goal_met": check.goal_met, "confidence": check.confidence}

## The router — three exits

Success, budget exhausted, or keep going. The middle one is what separates this from a toy.

In [ ]:
from langgraph.graph import END, START, StateGraph

CONFIDENCE_THRESHOLD = 7


def goal_router(state: AutoState) -> str:
    # Exit 1 - SUCCESS: the verdict AND the confidence.
    if state["goal_met"] and state["confidence"] >= CONFIDENCE_THRESHOLD:
        print("goal achieved")
        return "report"

    # Exit 2 - BUDGET EXHAUSTED: report honestly, do NOT pretend success.
    if state["iterations"] >= state["max_iterations"]:
        print("budget exhausted - reporting partial results")
        return "report"

    return "observe"      # another cycle


def report(state: AutoState) -> dict:
    """Produce the final output. Honesty about failure is mandatory."""
    status = "GOAL ACHIEVED" if state["goal_met"] else "GOAL NOT FULLY ACHIEVED (budget exhausted)"
    findings = "\n\n".join(state["findings"])

    text = llm.invoke(
        f"GOAL: {state['goal']}\nSTATUS: {status}\n"
        f"Iterations used: {state['iterations']}/{state['max_iterations']}\n\n"
        f"EVIDENCE:\n{findings}\n\n"
        "Write the final report in under 250 words. If the goal was NOT achieved, say so "
        "explicitly at the top and list exactly what is still missing. Do not paper over gaps."
    ).content
    return {"final_report": text}

In [ ]:
b = StateGraph(AutoState)

b.add_node("observe", observe)
b.add_node("plan", plan)
b.add_node("act", act)
b.add_node("evaluate", evaluate)
b.add_node("report", report)

b.add_edge(START, "observe")
b.add_edge("observe", "plan")
b.add_edge("plan", "act")
b.add_edge("act", "evaluate")
b.add_conditional_edges("evaluate", goal_router, ["observe", "report"])   # loop + exits
b.add_edge("report", END)

auto_graph = b.compile()
show(auto_graph)

## Give it a goal and walk away

In [ ]:
result = auto_graph.invoke(
    {
        "goal": "Determine whether Kafka, RabbitMQ or Pulsar is best for a "
                "high-throughput event backbone.",
        "success_criteria": [
            "Throughput figures for all three systems",
            "Latency characteristics for all three systems",
            "Durability model for all three systems",
            "A clear recommendation with justification",
        ],
        "findings": [],
        "iterations": 0,
        "max_iterations": 3,       # the budget
        "goal_met": False,
        "confidence": 0,
    },
    config={"recursion_limit": 50},    # the backstop
)

print("\n" + "=" * 60)
print(f"iterations: {result['iterations']} | goal_met: {result['goal_met']} "
      f"| confidence: {result['confidence']}/10\n")
print(result["final_report"])

### The failure exit is not hypothetical

Same graph, same budget, but a goal the knowledge base **cannot** satisfy — there's nothing
in there about pricing. A toy agent would confabulate numbers and declare victory. I want to
see it burn the budget and then say so.

In [ ]:
impossible = auto_graph.invoke(
    {
        "goal": "Determine the exact 2026 enterprise licence pricing for Kafka, RabbitMQ and Pulsar.",
        "success_criteria": [
            "Exact annual licence cost for each of the three systems",
            "A named vendor quote as the source for each figure",
        ],
        "findings": [],
        "iterations": 0,
        "max_iterations": 2,
        "goal_met": False,
        "confidence": 0,
    },
    config={"recursion_limit": 50},
)

print("\n" + "=" * 60)
print(f"goal_met: {impossible['goal_met']} (expected False)\n")
print(impossible["final_report"][:900])

# Part B — budget guardrails

Cost here is unbounded by construction, so this is not optional. Use **all** the layers, not
one — each catches a different runaway mode.

| layer | mechanism | catches |
|---|---|---|
| 1. iterations | `max_iterations` in state | no logical progress |
| 2. graph steps | `config={"recursion_limit": N}` | a routing bug |
| 3. wall clock | `started_at` in state | a slow/hanging tool |
| 4. tokens | count in state, or LangSmith limits | expensive prompts |
| 5. side effects | `interrupt_before=[...]` | irreversible actions |
| 6. tool scope | just don't hand it `send_email` / `delete_*` | everything else |

In [ ]:
import time


class BudgetState(AutoState):
    started_at: float
    max_seconds: float
    tokens_used: int
    max_tokens: int


def budget_router(state: BudgetState) -> str:
    """Layered defence, cheapest check first."""
    if state["goal_met"] and state["confidence"] >= CONFIDENCE_THRESHOLD:
        return "report"
    if state["iterations"] >= state["max_iterations"]:
        print("stop: iteration budget")
        return "report"
    if time.time() - state["started_at"] > state["max_seconds"]:
        print("stop: time budget")
        return "report"
    if state["tokens_used"] >= state["max_tokens"]:
        print("stop: token budget")
        return "report"
    return "observe"


# Quick check that each rail actually fires, without burning real API calls:
base = {
    "goal_met": False, "confidence": 0, "iterations": 0, "max_iterations": 5,
    "started_at": time.time(), "max_seconds": 60, "tokens_used": 0, "max_tokens": 10_000,
}
print("nothing exhausted   ->", budget_router({**base}))
print("iterations spent    ->", budget_router({**base, "iterations": 5}))
print("clock spent         ->", budget_router({**base, "started_at": time.time() - 120}))
print("tokens spent        ->", budget_router({**base, "tokens_used": 10_000}))
print("goal met            ->", budget_router({**base, "goal_met": True, "confidence": 9}))

# Part C — human-in-the-loop on dangerous actions

"Autonomous" is a spectrum, not a binary:

```
supervised ---------------------------------> fully autonomous
approve every step   approve risky steps      report at the end
   (safe, slow)         <- sweet spot          (fast, risky)
```

Most production "autonomous" agents sit in the middle: they loop freely on **read-only**
tools and interrupt for **writes**.

`interrupt()` pauses the graph and surfaces a payload to the caller. It **requires** a
checkpointer — the paused state has to be stored somewhere to be resumed.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt


class ApprovalState(TypedDict):
    goal: str
    action: str
    approved: bool
    result: str


def propose(state: ApprovalState) -> dict:
    """The agent decides what it wants to do (read-only reasoning)."""
    return {"action": f"send_email(to='cto@corp.com', subject='{state['goal']}')"}


def human_gate(state: ApprovalState) -> dict:
    """interrupt() PAUSES here. Execution resumes only on Command(resume=...)."""
    decision = interrupt({"question": "Approve this action?", "action": state["action"]})
    return {"approved": decision == "approve"}


def execute(state: ApprovalState) -> dict:
    if not state["approved"]:
        return {"result": "REJECTED by human - action NOT executed."}
    return {"result": f"EXECUTED: {state['action']}"}


ab = StateGraph(ApprovalState)
ab.add_node("propose", propose)
ab.add_node("human_gate", human_gate)
ab.add_node("execute", execute)
ab.add_edge(START, "propose")
ab.add_edge("propose", "human_gate")
ab.add_edge("human_gate", "execute")
ab.add_edge("execute", END)

approval_graph = ab.compile(checkpointer=InMemorySaver())

In [ ]:
# Run 1 -> hits interrupt() and stops
cfg = {"configurable": {"thread_id": "approval-1"}}
out = approval_graph.invoke({"goal": "Q3 architecture review"}, cfg)
print("PAUSED. the agent wants to:", out["__interrupt__"][0].value["action"])

# The human decides. Command(resume=X) makes interrupt() RETURN X.
print("rejected ->", approval_graph.invoke(Command(resume="reject"), cfg)["result"])

# Same graph, approved this time
cfg2 = {"configurable": {"thread_id": "approval-2"}}
approval_graph.invoke({"goal": "Q3 architecture review"}, cfg2)
print("approved ->", approval_graph.invoke(Command(resume="approve"), cfg2)["result"])

## Notes to self

**"Isn't this just ReAct with a while-loop?"** The mechanics look similar; the contract is
completely different:

| | ReAct | autonomous |
|---|---|---|
| stop signal | LLM emits no tool_calls | the **evaluator** says the criteria are met |
| progress tracked | no | yes, explicitly |
| can it fail? | it just returns something | it **reports failure honestly** |
| budget | `recursion_limit` (a crash) | a **first-class state field** |

**Failure modes:**

| symptom | cause | fix |
|---|---|---|
| declares success immediately | LLM optimism | strict evaluator + confidence threshold |
| never terminates | criteria are unachievable | `max_iterations` + an honest failure report |
| repeats the same action | the plan ignores findings | feed `observations` into the planner |
| burns budget on nothing | no progress detection | track findings-per-iteration, abort if flat |
| hallucinated findings | no grounding | tools only; forbid inventing facts in the prompt |
| costs explode | only one guardrail | layer all six |
| did something irreversible | write tools and no gate | `interrupt_before` on every side effect |

**When NOT to build this:** if success can't be defined *before* the run, don't. Use
Planning (03) plus a human in the loop instead — an agent with no measurable finish line
never finishes, it just runs out of money.

**API I used:**

```python
llm.with_structured_output(GoalCheck, method="json_schema")   # the verdict
Annotated[List[str], operator.add]                           # accumulate findings
add_conditional_edges("evaluate", router, ["observe", "report"])
config={"recursion_limit": 50}                               # backstop
interrupt({...})                                             # pause (needs a checkpointer)
graph.invoke(Command(resume=value), cfg)                     # resume
compile(interrupt_before=["execute"])                        # static gate
```

## The whole series in one picture

```
                        +- Reactive (1) ---- state -> action, no loop
                        |
   add a CYCLE ---------+- ReAct (2) ------- discover the path while acting
                        |
   add a PLAN ----------+- Planning (3) ---- decide the path upfront, replan
                        |
   add a CRITIC --------+- Reflective (4) -- judge & revise the OUTPUT
                        |
   add PERSISTENCE -----+- Memory (5) ------ identity across runs (augments the rest)
                        |
   add more AGENTS -----+- Multi-agent (6) - many decision makers
                        |
   add a GOAL + BUDGET -+- Autonomous (7) -- the agent owns the loop
```

Everything with a marketing name is a recombination of these:

| "X agent" | = |
|---|---|
| coding agent | Planning + Tools + Memory + **Self-Correction** |
| research agent | ReAct/Planning + RAG + Tools + Memory |
| AI assistant | ReAct + Tools + **Memory** |
| agentic RAG | ReAct/Planning + Retrieval |
| SWE agent | Planning + Reflection + Tools + an exec sandbox |
| deep research | **Autonomous** + Multi-agent + Reflection + Memory |

**Build order that actually works:**

```
1. Start with ReAct (2).      It solves most real problems.
2. Quality issues?          -> add Reflection (4).
3. Losing the thread?       -> add Planning (3).
4. Forgetting the user?     -> add Memory (5).
5. Too many tools?          -> split into Multi-agent (6).
6. Runs unattended?         -> wrap in Autonomous (7) + guardrails.
```

**Never start at 7.** Every step up multiplies cost, latency and debugging pain. Earn each one.